In [1]:
!pip install pdf2image Pillow chromadb torch torchvision torchaudio transformers librosa ipython open-clip-torch qwen_vl_utils

In [2]:
!apt-get update
!apt-get install -y poppler-utils

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,027 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [3]:
!pip install -U transformers

In [4]:
import os
from pdf2image import convert_from_path
from PIL import Image
import chromadb
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
import torch
from transformers import CLIPProcessor, CLIPModel, WhisperProcessor, WhisperForConditionalGeneration, AutoModelForVision2Seq, AutoProcessor # Changed to AutoModelForVision2Seq, AutoProcessor
import librosa
from sentence_transformers import SentenceTransformer
from qwen_vl_utils import process_vision_info
from IPython.display import display, Image as IPImage

ImportError: cannot import name 'AutoModelForVision2Seq' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
# logging.basicConfig(level=logging.INFO)
# logger = logging.getLogger(__name__)

In [ ]:
!mkdir dataset

In [ ]:
output_dir = "dataset"
image_output_dir = "extracted_images"

def convert_pdfs_to_images(folder, image_output_dir):
    if not os.path.exists(image_output_dir):
        os.makedirs(image_output_dir)

    pdf_files = [f for f in os.listdir(folder) if f.endswith('.pdf')]
    all_images = {}

    for doc_id, pdf_file in enumerate(pdf_files):
        pdf_path = os.path.join(folder, pdf_file)
        images = convert_from_path(pdf_path, dpi=100)

        image_paths = []
        for i, image in enumerate(images):
            image_path = os.path.join(image_output_dir, f"{doc_id}_page_{i}.png")
            image.save(image_path, "PNG")
            image_paths.append(image_path)

        all_images[doc_id] = image_paths
    return all_images

all_images = convert_pdfs_to_images(output_dir, image_output_dir)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def embed_images(image_paths):
    embeddings = []
    for path in image_paths:

        image = Image.open(path)

        inputs = processor(images=image, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            # Access the 'pooler_output' attribute as get_image_features is returning a BaseModelOutputWithPooling object
            image_embedding = model.get_image_features(**inputs).pooler_output.cpu().numpy()
        embeddings.append(image_embedding)
    return embeddings

image_embeddings = {}
for doc_id, paths in all_images.items():
    image_embeddings[doc_id] = embed_images(paths)

In [ ]:
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)

def transcribe_audio(audio_path, chunk_length=30):
    audio, sr = librosa.load(audio_path, sr=16000)
    chunk_size = chunk_length * sr
    chunks = [audio[i:i + chunk_size] for i in range(0, len(audio), chunk_size)]

    transcription_chunks = []
    for chunk in chunks:
        inputs = whisper_processor(chunk, sampling_rate=sr, return_tensors="pt").to(device)
        inputs["attention_mask"] = torch.ones_like(inputs.input_features)
        with torch.no_grad():
            predicted_ids = whisper_model.generate(**inputs, max_length=448)
        chunk_transcription = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        transcription_chunks.append(chunk_transcription)

    full_transcription = " ".join(transcription_chunks)
    return full_transcription, transcription_chunks

audio_files = [f for f in os.listdir(output_dir) if f.endswith('.mp3')]
audio_transcriptions = {}
for audio_id, audio_file in enumerate(audio_files):
    audio_path = os.path.join(output_dir, audio_file)
    full_transcription, transcription_chunks = transcribe_audio(audio_path)
    audio_transcriptions[audio_id] = {
        "full_transcription": full_transcription,
        "chunks": transcription_chunks
    }

In [ ]:
client = chromadb.PersistentClient(path="chroma_db")
embedding_function = OpenCLIPEmbeddingFunction()
text_embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Delete existing collections (if needed)
try:
    client.delete_collection(name="image_collection")
    client.delete_collection(name="audio_collection")
    print("Deleted existing collections.")
except Exception as e:
    print(f"Collections do not exist or could not be deleted: {e}")

image_collection = client.create_collection(name="image_collection", embedding_function=embedding_function)
audio_collection = client.create_collection(name="audio_collection")

for doc_id, embeddings in image_embeddings.items():
    for i, embedding in enumerate(embeddings):
        image_collection.add(
            ids=[f"image_{doc_id}_{i}"],
            embeddings=[embedding.flatten().tolist()],
            metadatas=[{"doc_id": str(doc_id), "image_path": all_images[doc_id][i]}]
        )

for audio_id, transcription_data in audio_transcriptions.items():
    transcription_chunks = transcription_data["chunks"]
    for chunk_id, chunk in enumerate(transcription_chunks):
        chunk_embedding = text_embedding_model.encode(chunk)
        audio_collection.add(
            ids=[f"audio_{audio_id}_chunk_{chunk_id}"],
            embeddings=[chunk_embedding.tolist()],
            metadatas=[{
                "audio_id": str(audio_id),
                "audio_path": audio_files[audio_id],
                "chunk_id": str(chunk_id)
            }],
            documents=[chunk]
        )

In [ ]:
def retrieve_data(query, top_k=2):

    query_embedding_image = embedding_function([query])[0]  # OpenCLIP for image collection
    query_embedding_audio = text_embedding_model.encode(query)  # SentenceTransformer for audio collection

    image_results = image_collection.query(
        query_embeddings=[query_embedding_image],
        n_results=top_k
    )

    audio_results = audio_collection.query(
        query_embeddings=[query_embedding_audio.tolist()],
        n_results=top_k
    )

    retrieved_images = [metadata["image_path"] for metadata in image_results["metadatas"][0] if "image_path" in metadata]
    retrieved_chunks = audio_results["documents"][0] if "documents" in audio_results else []

    return retrieved_images, retrieved_chunks

query = "What are the healthiest ingredients to use in recipe you have?"
retrieved_images, retrieved_chunks = retrieve_data(query)
print("Retrieved Images:", retrieved_images)
print("Retrieved Audio Chunks:", retrieved_chunks)

In [ ]:
# vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
# ).cuda().eval()

# min_pixels = 256 * 256
# max_pixels = 1024 * 1024
# vl_model_processor = Qwen2VLProcessor.from_pretrained(
#     "Qwen/Qwen2-VL-7B-Instruct",
#     min_pixels=min_pixels,
#     max_pixels=max_pixels
# )

In [ ]:
vl_model = AutoModelForVision2Seq.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", # Loading the 3B Vision-Language Instruct model
    torch_dtype=torch.bfloat16,
).cuda().eval()

min_pixels = 256 * 256
max_pixels = 1024 * 1024
vl_model_processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", # Loading the 3B Vision-Language Instruct model
    min_pixels=min_pixels,
    max_pixels=max_pixels
)

In [ ]:
from huggingface_hub import list_models
import pandas as pd

# Search for Qwen Vision-Language models
# This will list models with 'Qwen' in their ID and attempt to filter for VLM-related tags or IDs

all_qwen_models = list(list_models(search="Qwen"))

filtered_vl_models = []
for model_info in all_qwen_models:
    model_id = model_info.modelId
    tags = model_info.tags if model_info.tags else []

    # Heuristic filtering for Vision-Language models
    if ("vl" in model_id.lower() or "vision-language" in tags or "vlm" in model_id.lower()):
        filtered_vl_models.append({
            "model_id": model_id,
            "tags": ", ".join(tags),
            "downloads": model_info.downloads,
            "likes": model_info.likes,
        })

if filtered_vl_models:
    df = pd.DataFrame(filtered_vl_models)
    # Sort by downloads to prioritize more popular/stable models
    df = df.sort_values(by="downloads", ascending=False).reset_index(drop=True)
    print("Hugging Face에서 찾은 Qwen Vision-Language 모델 후보:")
    print(df)
    print("\nCPU 메모리를 가장 적게 사용하는 모델을 찾으려면, `model_id` 또는 설명에서 1.5B, 4B 등 더 작은 파라미터 수를 가진 모델을 확인하세요.\n하지만, 이전 시도에서 공식 `Qwen2-VL`의 7B보다 작은 모델은 일반적으로 사용되는 명명 규칙으로 찾기 어려웠습니다.\n\n만약 목록에서 적합한 더 작은 모델을 찾으시면, 위에 있는 셀(`ZmMRX_bS4oiq`)의 `Qwen/Qwen2-VL-7B-Instruct`를 해당 `model_id`로 교체하여 실행하시면 됩니다.")
else:
    print("지정된 조건에 맞는 Qwen Vision-Language 모델을 찾을 수 없습니다.")

In [ ]:
chat_template = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": retrieved_images[0]},  # First retrieved image
            {"type": "image", "image": retrieved_images[1]},  # Second retrieved image
            {"type": "text", "text": query},  # User query
            {"type": "text", "text": "Audio Context: " + " ".join(retrieved_chunks)}  # Include audio data
        ],
    }
]

text = vl_model_processor.apply_chat_template(
    chat_template, tokenize=False, add_generation_prompt=True
)

image_inputs, _ = process_vision_info(chat_template)
inputs = vl_model_processor(
    text=[text],
    images=image_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

generated_ids = vl_model.generate(**inputs, max_new_tokens=100)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = vl_model_processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])

In [ ]:
from PIL import Image
from IPython.display import display

# Function to display retrieved images
def display_retrieved_images(retrieved_images):
    for image_path in retrieved_images:
        try:
            # Open the image using PIL
            img = Image.open(image_path)
            print(f"Displaying image: {image_path}")
            display(img)  # Display the image in a notebook or GUI
        except Exception as e:
            print(f"Error displaying image {image_path}: {e}")


In [ ]:
display_retrieved_images(retrieved_images)